In [5]:
import math
import random

import torch
import numpy as np
import torch.nn as nn
from collections import Counter
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

# DEVICE = torch.device("xpu" if torch.xpu.is_available() else "cpu")
DEVICE = "cpu"
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
#✔️

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
#✔️
train_full = datasets.CIFAR10(root="data", train=True, download=True, transform=image_transform)
test_full = datasets.CIFAR10(root="data", train=False, download=True, transform=image_transform)
#✔️
TRAIN_LIMIT = 4096
TEST_LIMIT = 1000

train_dataset = Subset(train_full, range(TRAIN_LIMIT))
test_dataset = Subset(test_full, range(TEST_LIMIT))
#✔️
labels = [test_full.targets[i] for i in test_dataset.indices]
class_counts = Counter(labels)
#✔️
print(class_counts)
print(type(class_counts))

class_names = test_full.classes

for class_idx, count in sorted(class_counts.items()):
    print(f"{class_names[class_idx]}: {count}")

print("Min:", min(class_counts.values()), "Max:", max(class_counts.values()))

Counter({6: 112, 9: 109, 8: 106, 3: 103, 0: 103, 7: 102, 2: 100, 4: 90, 1: 89, 5: 86})
<class 'collections.Counter'>
airplane: 103
automobile: 89
bird: 100
cat: 103
deer: 90
dog: 86
frog: 112
horse: 102
ship: 106
truck: 109
Min: 86 Max: 112


In [6]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)
#✔️
class_names = train_full.classes
print(f"classes: {class_names}")

images, labels = next(iter(train_loader))
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")

classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
images shape: torch.Size([64, 3, 32, 32])
labels shape: torch.Size([64])


Question 0.1:

<div style="direction: rtl; text-algin: right;">
generalization بهتر، فدای زمان پردازش می‌شود
</div>

Question A1:
<div style="direction: rtl; text-algin: right;">
با توجه به چند کلاسه بودن مسئله، استفاده از cross-entropy گزینه مناسب تری است
</div>

In [7]:
class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        self.network =  nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 10)
        )
        # self.network = nn.Sequential(nn.Linear(32 * 32 * 3, 10), nn.ReLU())

    def forward(self, x):
        return self.network(x)

In [8]:
toy_model = ToyModel().to(device=DEVICE)

logits = toy_model(images.to(device="cpu"))
print(logits.shape)
print(logits[0].argmax())
print(labels[0])


# 64 = Batch size 10 = N(classes)

# If Batch size was 32 the first dimension will be 32 and the second dimension will be 10

torch.Size([64, 10])
tensor(8)
tensor(0)


In [9]:
# calculate loss with torch
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
print(f"loss torch {loss}")
# By hand
probabilites = torch.softmax(logits, dim=1).detach().numpy()

loss_manual = np.mean(-(np.log(probabilites[torch.arange(len(labels)),labels])))
print(f"loss torch {loss_manual}")
print(f"log 1/10 : {- np.log(0.1)}")
print(f"log 1/100 : {- np.log(0.01)}")
print(f"log 9/10 : {- np.log(0.9)}")

# هر وقت مدل با اطمینان زیاد اشتباه پیش بینی کند یا احتمال کمی برای کلاس درست در نظر بگیرد لاس جریمه بزرگی برای آن در نظر میگیرد

loss torch 2.3087856769561768
loss torch 2.3087856769561768
log 1/10 : 2.3025850929940455
log 1/100 : 4.605170185988091
log 9/10 : 0.10536051565782628


# non-Safety

In [10]:
# X
generator = torch.Generator().manual_seed(42)

logit = torch.randn((100,10),dtype=torch.float,generator=generator).to(device=DEVICE)*50
target = torch.zeros(size=(100,),dtype=torch.long).to(DEVICE)

P = torch.exp(logit)/torch.sum(torch.exp(logit), dim=1, keepdim=True)

loss = torch.mean(-torch.log(P))
print(f"loss torch {loss}")
print(f"loss torch one row {-torch.log(P)[10]}\n-----------------")


print(f"Controled loss with nn.CrossEntropyLoss() = {criterion(logit, target)}")

loss torch nan
loss torch one row tensor([3.1621e+01, 3.4496e+01, 1.0312e+01, 8.6166e+01, 7.2922e+01, 2.6021e+01,
        4.3421e+01, 5.4735e+01, 5.0413e+01, 3.3260e-05])
-----------------
Controled loss with nn.CrossEntropyLoss() = 87.07770538330078


In [11]:
generator = torch.Generator().manual_seed(42)
logit = torch.randn((100,10),dtype=torch.float,generator=generator).to(device=DEVICE)*50
target = torch.zeros(size=(100,),dtype=torch.long).to(DEVICE)

# maximum logits
m = torch.max(logit, dim=1, keepdim=True).values


target_logits = logit[torch.arange(len(target)), target].unsqueeze(1)
term2 =m + torch.log(torch.sum(torch.exp(logit-m),dim=1 ,keepdim=True))
loss= -( target_logits - term2)
print(f"calculate logexpsum by hand : {torch.mean(loss)}")
print(f"by torch {criterion(logit, target)}")
func_loss = - (target_logits - (m + torch.logsumexp(logit-m, dim=1)))
# loss = torch.logsumexp(logit, dim=1)
print(f"with logsum torch : {torch.mean(func_loss)}")

calculate logexpsum by hand : 87.07771301269531
by torch 87.07770538330078
with logsum torch : 87.07771301269531


In [12]:
# [B, 3, 32, 32]
#[N+2p-k/stride  ]+1
#[32+2-3/1] + 1 = 32
# layer1: [B, 16, 32, 32] -> Relu -> MaxPool(kernel=2) -> [B, 16, 16, 16]
# layer2: [B, 32, 16, 16] -> Relu -> MaxPool(kernel=2) -> [B, 32, 8, 8]
# layerLinear:  flatten:[B,2048] -> Linear:[B,10] ==>10classes

In [13]:
class SmallCnn(nn.Module):
    #✔️
    def __init__(self,num_classes:int):
        #✔️
        super().__init__()
        #✔️
        self.num_classes = int(num_classes)
        #Layer1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3,padding=1)
        #Layer2
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3,padding=1)
        # fully connected layers
        self.fc = nn.Linear(2048,64)
        self.fc2 = nn.Linear(64, self.num_classes)


        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        #✔️


    def forward(self,x):
        #Layer1
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        #✔️
        #Layer2
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        #✔️
        #FullyConnected1
        x = self.flatten(x)
        x = self.fc(x)
        x = self.relu(x)
        #✔️
        # FullyConnected2
        x = self.fc2(x)
        return x
        #✔️






In [14]:
model = SmallCnn(num_classes=len(class_names)).to(device=DEVICE)
#✔️
criterion = nn.CrossEntropyLoss()
#✔️
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#✔️



In [15]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    #✔️

    # loss_history_batch = []

    total_loss = 0
    total_correct = 0
    total_examples = 0
    #✔️
    for images, labels in train_loader:
        #✔️
        images, labels = images.to(device=device), labels.to(device=device)
        #✔️
        logits = model(images)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        #✔️
        # loss_history_batch.append(loss.item())
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples +=batch_size

    loss_history = total_loss/total_examples
    accuracy_history=total_correct/total_examples

    return loss_history, accuracy_history


In [16]:
def evaluate(model,test_loader,criterion,device):
    model.eval()

    # loss_history_batch = []

    total_example=0
    total_loss = 0
    total_correct = 0
    with torch.no_grad():
        for images, labels in test_loader:

            images, labels = images.to(device=device), labels.to(device=device)
            logits = model(images)
            loss = criterion(logits, labels)
            # loss_history_batch.append(loss.item())
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_example += batch_size
    loss_history =total_loss/total_example
    accuracy_history = total_correct/total_example

    return loss_history, accuracy_history

In [17]:
EPOCHS =2
loss_history = []
accuracy_history = []
loss_val_history = []
accuracy_val_history = []

for epoch in range(EPOCHS):
    # train model
    loss,accuracy = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    loss_history.append(loss)
    accuracy_history.append(accuracy)
    # validation
    loss_val , accuracy_val = evaluate(model,test_loader,criterion,DEVICE)
    loss_val_history.append(loss_val)
    accuracy_val_history.append(accuracy_val)
    print(f"epoch = {epoch+1} | {"train= > loss":20} = {loss:.4f}, accuracy = {accuracy:.4f}")
    print(f"epoch =   | {"validation= > loss":20} = {loss_val:.4f}, accuracy = {accuracy_val:.4f}\n{"-"*40}")





epoch = 1 | train= > loss        = 2.0492, accuracy = 0.2664
epoch =   | validation= > loss   = 1.8551, accuracy = 0.3500
----------------------------------------
epoch = 2 | train= > loss        = 1.7415, accuracy = 0.3840
epoch =   | validation= > loss   = 1.6547, accuracy = 0.4160
----------------------------------------


In [18]:
# B2
# 1 => RunTime Error
# 2 => گرادیان ها جمع میشود از دور قبلی
# 3 => در حالت آموزش روی حالت .train() قرار دادیم ولی در حالت ارزیابی روی حالت  .eval() تا گرادیان محاسبه نکند و در بلاک torch.no_grad()
# بچ ها رو لود کردیم تا گرادیان محاسبه نکند ! قرار دادن مدل در حالت ارزیابی و محاسبه نکردن گرادیان موجب افزایش توان مدل و غیر فعال شدن آپدیت رانینگ مین و واریانس است و استفاده نکردن از گرادیان باعث افزایش حافظه و سرعت پردازش میشود

# چون به صورتی داخلی احتمال رو حساب میکند و نیاز به احتمال سافت مکس قبل از دادن به تابع هزینه نیست

# بله لاس ما اندکی بهتر ازاین حالت است
print (f"log 1/10 {-(np.log(0.1)):.3f}")

log 1/10 2.303


# B3 Work IN Process